# Face & Object Detection

---

## Learning Objectives

Students will learn:
- The theory behind Haar Cascades and Pre-trained AI models.
- How to use `detectMultiScale` to scan an image for objects.
- How to define a Region of Interest (ROI) to speed up processing.
- How to detect faces and smiles in real-time.

---

## Prerequisites
- Deep understanding of Grayscale conversion.
- Understanding of Webcam integration (`cv2.VideoCapture`).
- Understanding of NumPy array slicing (for ROI).

---

## Dataset / Assets Used
- Live webcam feed (Camera Index 0).
- Haar Cascade XML files (downloaded from OpenCV GitHub).

## Import Libraries
Let's start by importing the necessary libraries.

In [ ]:
import cv2
import numpy as np

## Verify OpenCV Installation

In [ ]:
print(f"OpenCV Version: {cv2.__version__}")

## The Concept: Haar Cascades

Before modern AI, computers didn't know what a "face" was. Haar Cascades are pre-trained AI detectors.
- **Pre-Trained:** They know what faces look like because they were trained on thousands of positive and negative images.
- **Fast & Lightweight:** They work perfectly in real-time.
- **Offline:** No internet required, runs locally.

We use XML files containing the "AI Brains":
- `haarcascade_frontalface_default.xml`
- `haarcascade_eye.xml`
- `haarcascade_smile.xml`


## The `detectMultiScale` Function

This function actually scans your image to find the face.
**Syntax:** `cascade.detectMultiScale(image, scaleFactor, minNeighbors)`

- **`scaleFactor` (Zoom Level):** Scales the image down in passes to find faces of different sizes.
  - `1.1` = 10% reduction. Standard, balanced choice.
  - `1.05` = 5% reduction. Extremely accurate, but slow.
- **`minNeighbors` (Confidence Check):** How many strict checks it needs before confirming a face.
  - `3` = Loose. Might detect random objects (false positives).
  - `5` = Safe checking. The sweet spot.


## Region of Interest (ROI)

If you are looking for eyes or a smile, **do not search the whole background**.
Once OpenCV finds the `(X, Y, Width, Height)` of a face, we use NumPy slicing to crop that box out. We then tell the eye and smile detectors to *only* search inside that cropped box (the ROI).


In [ ]:
# 1. Load the Pre-Trained AI Models (XML files)
# Note: Ensure these XML files are in your working directory!
# If you don't have them, you can use cv2.data.haarcascades to get default paths.

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
eye_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_eye.xml")
smile_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_smile.xml")

# 2. Start the Webcam
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break
        
    # 3. Convert frame to Grayscale (Crucial for Haar Cascades)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    # 4. Detect Faces: scaleFactor=1.1, minNeighbors=5
    faces = face_cascade.detectMultiScale(gray, 1.1, 5)
    
    # 5. Loop through every face detected
    for (x, y, w, h) in faces:
        
        # A. Draw a green rectangle around the main face
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
        
        # B. Define the Region of Interest (ROI) for Grayscale (detection) and Color (drawing)
        roi_gray = gray[y:y+h, x:x+w]
        roi_color = frame[y:y+h, x:x+w]
        
        # C. Detect Eyes inside the ROI
        eyes = eye_cascade.detectMultiScale(roi_gray, 1.1, 10)
        if len(eyes) > 0:
            cv2.putText(frame, "Eyes Detected", (x, y - 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
            
        # D. Detect Smiles inside the ROI
        # Smiles are trickier: bigger scaleFactor(1.7) and stricter minNeighbors(20)
        smiles = smile_cascade.detectMultiScale(roi_gray, 1.7, 20)
        if len(smiles) > 0:
            cv2.putText(frame, "Smiling!", (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
            
    # 6. Show output
    cv2.imshow("Smart Face Detector", frame)
    
    # 7. Quit logic
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

## Common Mistakes
> **Warning**  
> - Forgetting to convert the frame to Grayscale before passing it to `detectMultiScale`. It will crash.
> - Slicing the ROI backward. Remember NumPy slicing is `[y_start:y_end, x_start:x_end]`.
> - Not verifying that the `.xml` files successfully loaded. If they are in the wrong folder, `CascadeClassifier` won't crash immediately, but `detectMultiScale` will fail later.


## Key Takeaways
- Haar Cascades are fast, lightweight, pre-trained AI models for object detection.
- `scaleFactor` and `minNeighbors` are the key tuning parameters to balance speed vs accuracy.
- Using ROI (Region of Interest) massively improves performance by limiting the search area.


## Practice Exercises
1. Change `minNeighbors` for the face detection to `1`. What happens? (You should see lots of false positives).
2. Draw rectangles around the eyes inside the `roi_color` loop.
3. Build an Emotion Tracker: If `len(smiles) > 0` print "Good Mood", else print "Neutral".
4. Experiment with `haarcascade_mcs_eyepair_small.xml` to detect both eyes at once or sunglasses.
5. Create a script that takes a photo (from the webcam) only when a smile is detected.


## Course Completed!
Congratulations, you have finished the OpenCV Basics course. You now have a solid foundation in image manipulation, drawing, filtering, structual analysis, and AI object detection!
